# Семинар 4. Математическое ожидание и дисперсия

**Курс «Теория вероятностей», ИТМО.** Лекция 4: математическое ожидание, дисперсия, стандартное отклонение, ковариация и корреляция.

Сегодня проверим все формулы лекции **симуляциями**, разберёмся, чем `np.var` отличается от `pd.Series.var`, и посмотрим на корреляции в реальных данных — включая случаи, когда они обманывают.

| Время | Блок | Что делаем |
|---|---|---|
| 0–5 мин | 0 | Запускаем ноутбук |
| 5–20 мин | 1 | Математическое ожидание: ЗБЧ, точный расчёт и симуляция, Коши |
| 20–35 мин | 2 | Линейность и метод индикаторов |
| 35–50 мин | 3 | Дисперсия: свойства, ловушка вычисления, `ddof` |
| 50–60 мин | 4 | Стандартное отклонение, z-оценки и неравенство Чебышёва |
| 60–70 мин | 5 | Среднее $n$ наблюдений: закон $\sqrt n$ |
| 70–90 мин | 6 | Ковариация и корреляция: портфель, квартет Энскомба, реальные данные |

Задания отмечены **✏️ Задание** и строкой `# TODO` в коде. Там, где стоит `...`, нужно дописать код.

## Как открыть ноутбук в Google Colab

1. Скачайте файл `04-seminar-expectation-variance-itmo.ipynb` на свой компьютер.
2. Откройте [colab.research.google.com](https://colab.research.google.com) и войдите в свой Google-аккаунт.
3. Выберите «Файл → Загрузить блокнот» (File → Upload notebook), перейдите на вкладку «Загрузить» и выберите скачанный файл. Можно просто перетащить файл в окно.
4. Ноутбук сохранится на вашем Google Диске в папке «Colab Notebooks». В следующий раз открывайте его оттуда, а не загружайте заново, иначе потеряете свои правки.
5. «Среда выполнения → Выполнить все» (Runtime → Run all) запускает все ячейки; одна ячейка запускается сочетанием `Shift+Enter`.

Ничего устанавливать не нужно: `numpy`, `scipy`, `pandas` и `matplotlib` в Colab уже есть.

## 0. Подготовка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats

rng = np.random.default_rng(2026)   # фиксируем генератор, чтобы результаты повторялись
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("numpy", np.__version__, "| scipy", scipy.__version__)

**Памятка.**

| Что | Теория | По выборке `x` |
|---|---|---|
| математическое ожидание | `stats.<distr>.mean()` | `x.mean()` |
| дисперсия | `.var()` | `np.var(x, ddof=1)` или `pd.Series(x).var()` |
| стандартное отклонение | `.std()` | `np.std(x, ddof=1)` |
| ковариация | — | `np.cov(x, y)[0, 1]` |
| корреляция | — | `np.corrcoef(x, y)[0, 1]`, `df.corr()` |

---
## 1. Математическое ожидание

### 1.1. Точный расчёт и закон больших чисел

Ряд распределения из лекции: $X$ принимает значения $0, 1, 2, 3, 6$ с вероятностями $0{,}1;\ 0{,}3;\ 0{,}3;\ 0{,}2;\ 0{,}1$. Посчитаем $EX=\sum x_ip_i$ и сравним со средним большой выборки.

In [ ]:
values = np.array([0, 1, 2, 3, 6])
probs = np.array([0.1, 0.3, 0.3, 0.2, 0.1])

EX = np.sum(values * probs)            # то же: values @ probs
sample = rng.choice(values, p=probs, size=100_000)
print(f"EX = {EX:.4f},  среднее выборки = {sample.mean():.4f}")

n = np.arange(1, 5001)
for _ in range(4):
    s = rng.choice(values, p=probs, size=n.size)
    plt.semilogx(n, np.cumsum(s) / n, lw=1)
plt.axhline(EX, color="k", ls="--", label="EX")
plt.xlabel("n"); plt.ylabel("среднее первых n значений"); plt.legend(); plt.show()

### ✏️ Задание 1.1. Максимум двух кубиков

$Y=\max(i,j)$ для пары кубиков. На семинаре 3 мы нашли $P(Y=k)=\dfrac{2k-1}{36}$.

1. Найдите $EY$ **точно** по ряду распределения (ответ — дробь $161/36\approx4{,}47$).
2. Найдите $EY^2$ и $DY$ точно.
3. Проверьте симуляцией 200 000 бросков.

In [ ]:
k = np.arange(1, 7)
pY = (2 * k - 1) / 36

# TODO: точные EY, EY^2, DY
EY = ...
EY2 = ...
DY = ...

# TODO: симуляция
dice = rng.integers(1, 7, size=(200_000, 2))
Y = ...

print(f"EY: точно {EY:.4f}, симуляция {Y.mean():.4f}")
print(f"DY: точно {DY:.4f}, симуляция {Y.var():.4f}")

### 1.2. Непрерывный случай: интеграл и Монте-Карло

$EX=\int x f(x)\,dx$ можно посчитать численно (`scipy.integrate.quad`) или как среднее большой выборки. Для плотности $f(x)=3x^2$ на $[0;1]$ точный ответ $3/4$.

In [ ]:
from scipy.integrate import quad

f = lambda x: 3 * x**2
EX_quad, _ = quad(lambda x: x * f(x), 0, 1)
X = rng.random(200_000) ** (1 / 3)          # метод обратной функции: F(x) = x^3
print(f"quad: {EX_quad:.5f},  Монте-Карло: {X.mean():.5f},  точно: {3/4}")

### 1.3. Когда ожидания нет: распределение Коши

Сравним скользящее среднее для нормального распределения и распределения Коши (`rng.standard_cauchy`).

In [ ]:
n = np.arange(1, 200_001)
plt.semilogx(n, np.cumsum(rng.normal(size=n.size)) / n, label="нормальное")
for _ in range(3):
    plt.semilogx(n, np.cumsum(rng.standard_cauchy(size=n.size)) / n, lw=1, label="Коши")
plt.ylim(-10, 10); plt.legend(); plt.xlabel("n"); plt.title("Скользящее среднее"); plt.show()

*Вопрос.* Медиана выборки из распределения Коши стабилизируется? Проверьте `np.median` для $n=10^3, 10^5, 10^6$. Почему медиане «не мешают» тяжёлые хвосты?

---
## 2. Линейность и метод индикаторов

### 2.1. Линейность работает и для зависимых величин

Возьмём **зависимые** $X$ и $Y=X^2+\text{шум}$. Проверим, что $E(X+Y)=EX+EY$, но $E(XY)\ne EX\cdot EY$.

In [ ]:
X = rng.exponential(2, size=500_000)
Y = X**2 + rng.normal(size=X.size)
print(f"E(X+Y) = {np.mean(X + Y):.3f},   EX + EY = {X.mean() + Y.mean():.3f}")
print(f"E(XY)  = {np.mean(X * Y):.3f},   EX · EY = {X.mean() * Y.mean():.3f}")

Z = rng.exponential(2, size=X.size)        # независимая копия
print(f"независимые: E(XZ) = {np.mean(X * Z):.3f},   EX · EZ = {X.mean() * Z.mean():.3f}")

### ✏️ Задание 2.1. Письма и конверты

$n$ писем случайно разложили по $n$ конвертам (случайная перестановка: `rng.permutation(n)`). Совпадение — письмо $i$ в конверте $i$.

1. Симулируйте 100 000 раскладок для $n=5$ и $n=50$, найдите среднее число совпадений. Теория: $1$ при любом $n$.
2. Найдите также **дисперсию** числа совпадений. Теория: тоже $1$ (при $n\ge2$). *Подсказка для доказательства дома:* $\mathrm{cov}(I_i,I_j)=\frac{1}{n(n-1)}-\frac1{n^2}$.
3. Сравните распределение числа совпадений с $\mathrm{Pois}(1)$.

*Подсказка:* для векторизации удобно `rng.permuted(np.tile(np.arange(n), (reps, 1)), axis=1)`.

In [ ]:
reps = 100_000
for n in [5, 50]:
    # TODO: матрица reps × n случайных перестановок и число совпадений в каждой строке
    matches = ...
    print(f"n = {n}: среднее {matches.mean():.4f}, дисперсия {matches.var():.4f}")

kk = np.arange(0, 7)
plt.bar(kk, [np.mean(matches == j) for j in kk], alpha=0.6, label="симуляция, n = 50")
plt.plot(kk, stats.poisson.pmf(kk, 1), "ko-", label="Pois(1)")
plt.legend(); plt.show()

### ✏️ Задание 2.2. Сколько разных значений?

В таблице $m$ записей, у каждой независимо и равновероятно один из $N$ идентификаторов категорий. Сколько **разных** категорий будет встречено в среднем?

1. Выведите формулу методом индикаторов: $I_j$ — категория $j$ встретилась хотя бы раз, $P(I_j=1)=1-(1-1/N)^m$.
2. Посчитайте `E_distinct_theory` для $N=100$, $m=150$ и проверьте симуляцией (`len(np.unique(...))`, 20 000 повторений).

Это же задача про лифт из вопросов для самопроверки: $N$ этажей, $m$ пассажиров.

In [ ]:
N, m = 100, 150

# TODO
E_distinct_theory = ...
E_distinct_sim = ...

print(f"теория {E_distinct_theory:.3f},  симуляция {E_distinct_sim:.3f}")

---
## 3. Дисперсия

### ✏️ Задание 3.1. Свойства дисперсии на симуляции

Пусть $X\sim\mathrm{Exp}(\lambda=0{,}5)$ (то есть $EX=2$, $DX=4$) и $U\sim U(0;6)$ независимы.

Посчитайте по выборкам объёма 1 000 000 и сравните с теорией:

1. $D(3X-7)$ — теория $9\cdot DX$;
2. $D(X+U)$ и $D(X-U)$ — теория $DX+DU$ в обоих случаях;
3. $D(X+X)$ — почему это **не** $DX+DX$? Чему равно на самом деле?

In [ ]:
X = rng.exponential(scale=2, size=1_000_000)
U = rng.uniform(0, 6, size=X.size)
DX, DU = 4, 36 / 12

# TODO: выборочные дисперсии и теоретические значения
rows = [
    ("D(3X − 7)", ..., ...),
    ("D(X + U)", ..., ...),
    ("D(X − U)", ..., ...),
    ("D(X + X)", ..., ...),
]
print(pd.DataFrame(rows, columns=["величина", "симуляция", "теория"]).round(3))

### 3.2. Ловушка формулы $EX^2-(EX)^2$

Время событий в миллисекундах от начала эпохи — числа порядка $1{,}7\cdot10^{12}$, а разброс всего несколько секунд. Посчитаем дисперсию двумя способами в `float64` и в `float32`.

In [ ]:
t = 1.7e12 + rng.normal(0, 1000, size=100_000)       # σ = 1000 мс, DX = 10^6

naive = lambda x: np.mean(x**2) - np.mean(x) ** 2
by_def = lambda x: np.mean((x - x.mean()) ** 2)
for dtype in [np.float64, np.float32]:
    x = t.astype(dtype)
    print(f"{dtype.__name__}: EX² − (EX)² = {naive(x):.4g},   E(X − EX)² = {by_def(x):.4g},   np.var = {np.var(x):.4g}")

*Вывод.* Формула «среднее квадрата минус квадрат среднего» хороша на бумаге. В коде считайте отклонения от среднего (или вычтите сначала любую близкую константу: $D(X-c)=DX$). А в `float32` (как часто бывает в GPU-вычислениях) точности не хватает уже для хранения самих значений: сдвиг на константу нужно сделать **до** перевода в `float32`.

### ✏️ Задание 3.2. Делить на $n$ или на $n-1$?

`np.var(x)` делит на $n$, `np.var(x, ddof=1)` и `pd.Series(x).var()` — на $n-1$.

1. Сгенерируйте 100 000 выборок объёма $n=5$ из $N(0, \sigma^2=4)$ (матрица `100_000 × 5`).
2. Для каждой выборки посчитайте обе оценки и усредните по выборкам.
3. Какая оценка в среднем равна $\sigma^2=4$? Чему равно среднее смещённой оценки? Сравните с формулой $\frac{n-1}{n}\sigma^2$.
4. Повторите для $n=50$: важна ли разница?

In [ ]:
for n in [5, 50]:
    samples = rng.normal(0, 2, size=(100_000, n))
    # TODO: средние значения двух оценок по всем выборкам
    mean_ddof0 = ...
    mean_ddof1 = ...
    print(f"n = {n}: ddof=0 → {mean_ddof0:.4f} (теория {(n - 1) / n * 4:.4f}),  ddof=1 → {mean_ddof1:.4f}")

---
## 4. Стандартное отклонение и неравенство Чебышёва

### ✏️ Задание 4.1. Насколько грубо неравенство Чебышёва?

Для четырёх распределений с конечной дисперсией найдите **точно** (через `scipy.stats`) вероятность $P(|X-EX|\ge k\sigma)$ при $k=1{,}5;\ 2;\ 3$ и сравните с границей $1/k^2$.

Распределения: $N(0,1)$, $U(0;1)$, $\mathrm{Exp}(1)$, $\mathrm{Pois}(2)$.

*Подсказка:* $P(|X-\mu|\ge k\sigma)=P(X\le\mu-k\sigma)+P(X\ge\mu+k\sigma)$. Для непрерывных это `cdf(μ − kσ) + sf(μ + kσ)`. Для дискретных аккуратно с границей: $P(X\ge a)$ = `sf(a − 1)` для целого $a$, или проще — просуммируйте `pmf` по нужным значениям.

In [ ]:
dists = {"N(0,1)": stats.norm(), "U(0;1)": stats.uniform(), "Exp(1)": stats.expon(), "Pois(2)": stats.poisson(2)}

def tail_prob(dist, k):
    mu, sd = dist.mean(), dist.std()
    # TODO: P(|X − mu| >= k·sd)
    ...

table = pd.DataFrame({name: [tail_prob(d, k) for k in [1.5, 2, 3]] for name, d in dists.items()},
                     index=["k = 1.5", "k = 2", "k = 3"])
table["Чебышёв 1/k²"] = [1 / k**2 for k in [1.5, 2, 3]]
table.round(4)

### 4.2. z-оценки и поиск аномалий

Время ответа сервиса часто распределено логнормально (лекция 3). Пометим как аномалии значения с $|z|>3$ и посмотрим, сколько их окажется, если на самом деле аномалий **нет**.

In [ ]:
latency = rng.lognormal(mean=np.log(120), sigma=0.6, size=100_000)   # мс, без аномалий
z = (latency - latency.mean()) / latency.std()
print(f"доля |z| > 3: {np.mean(np.abs(z) > 3):.4%}   (для нормального было бы 0,27 %)")
print(f"асимметрия {stats.skew(latency):.2f}, эксцесс {stats.kurtosis(latency):.2f}")

z_log = (np.log(latency) - np.log(latency).mean()) / np.log(latency).std()
print(f"после логарифма: доля |z| > 3 = {np.mean(np.abs(z_log) > 3):.4%}")

*Вопрос.* Почему после логарифмирования доля «аномалий» стала близка к 0,27 %? Какой вывод для мониторинга?

---
## 5. Среднее $n$ наблюдений: закон $\sqrt n$

$D\bar X=\sigma^2/n$, поэтому стандартное отклонение среднего равно $\sigma/\sqrt n$.

In [ ]:
ns = np.array([1, 4, 16, 64, 256, 1024])
sd_mean = [rng.exponential(1, size=(5_000, n)).mean(axis=1).std() for n in ns]
plt.loglog(ns, sd_mean, "o", ms=8, label="симуляция")
plt.loglog(ns, 1 / np.sqrt(ns), "k-", label="σ/√n")
plt.xlabel("n"); plt.ylabel("ст. отклонение среднего"); plt.legend(); plt.show()

### ✏️ Задание 5.1. Конверсия в A/B-тесте

Конверсия страницы $p=0{,}05$ (каждый посетитель покупает с вероятностью 5 %, независимо). Конверсию оценивают долей покупателей $\hat p$ среди $n$ посетителей.

1. Выведите формулу стандартного отклонения $\hat p$: $\sigma_{\hat p}=\sqrt{p(1-p)/n}$.
2. Найдите минимальное `n_needed`, при котором $\sigma_{\hat p}\le0{,}002$ (то есть $\pm0{,}2$ процентного пункта).
3. Проверьте симуляцией: 20 000 «экспериментов» с этим $n$ (`rng.binomial`), посчитайте стандартное отклонение $\hat p$.
4. Во сколько раз нужно увеличить $n$, чтобы погрешность стала $0{,}001$?

In [ ]:
p = 0.05

# TODO
n_needed = ...
p_hat = ...          # 20 000 оценок конверсии

print(f"n = {n_needed},  σ(p̂): теория {np.sqrt(p * (1 - p) / n_needed):.5f}, симуляция {p_hat.std():.5f}")

---
## 6. Ковариация и корреляция

### 6.1. Рассылка и покупка

Совместное распределение из лекции. Смоделируем 100 000 пользователей и сравним выборочные ковариацию и корреляцию с теоретическими $0{,}1$ и $0{,}408$.

In [ ]:
pairs = np.array([(0, 0), (0, 1), (1, 0), (1, 1)])
pp = np.array([0.4, 0.1, 0.2, 0.3])
idx = rng.choice(4, p=pp, size=100_000)
x, y = pairs[idx].T

print("cov:", np.cov(x, y)[0, 1], "  теория 0.1")
print("ρ:  ", np.corrcoef(x, y)[0, 1], "  теория", 0.1 / np.sqrt(0.25 * 0.24))
print("E(XY) − EX·EY:", np.mean(x * y) - x.mean() * y.mean())

### ✏️ Задание 6.1. Портфель из двух активов

Дневные доходности двух активов: $\sigma_X=0{,}02$, $\sigma_Y=0{,}03$, корреляция $\rho$. Портфель $P=aX+(1-a)Y$.

1. Напишите функцию `portfolio_sd(a, rho)` по формуле $D(aX+bY)=a^2DX+b^2DY+2ab\,\rho\,\sigma_X\sigma_Y$.
2. Постройте $\sigma_P(a)$ для $\rho\in\{-0{,}5;\ 0;\ 0{,}5;\ 0{,}9\}$.
3. Найдите долю `a_best`, минимизирующую риск при $\rho=0{,}5$ (численно, по сетке). *Аналитически:* $a^*=\dfrac{\sigma_Y^2-\rho\sigma_X\sigma_Y}{\sigma_X^2+\sigma_Y^2-2\rho\sigma_X\sigma_Y}$.
4. Проверьте ответ симуляцией: сгенерируйте коррелированные доходности через `rng.multivariate_normal` и посчитайте выборочное $\sigma_P$.

In [ ]:
sx, sy = 0.02, 0.03

def portfolio_sd(a, rho):
    # TODO
    ...

a = np.linspace(0, 1, 501)
for rho in [-0.5, 0, 0.5, 0.9]:
    plt.plot(a, portfolio_sd(a, rho), label=f"ρ = {rho}")
plt.xlabel("доля первого актива a"); plt.ylabel("σ портфеля"); plt.legend(); plt.show()

# TODO: a_best для rho = 0.5 и проверка симуляцией
a_best = ...
cov = [[sx**2, 0.5 * sx * sy], [0.5 * sx * sy, sy**2]]
R = rng.multivariate_normal([0, 0], cov, size=500_000)
sd_sim = ...
print(f"a* = {a_best:.3f},  σ: формула {portfolio_sd(a_best, 0.5):.5f}, симуляция {sd_sim:.5f}")

### ✏️ Задание 6.2. Квартет Энскомба

Четыре набора данных с **почти одинаковыми** средними, дисперсиями и корреляцией (Anscombe, 1973). Данные ниже.

1. Посчитайте для каждого набора: среднее и дисперсию `x` и `y` (`ddof=1`), корреляцию, наклон регрессии $b=\mathrm{cov}(x,y)/Dx$.
2. Постройте четыре диаграммы рассеяния.
3. Для какого набора корреляция Пирсона описывает связь адекватно? Посчитайте ещё корреляцию Спирмена (`stats.spearmanr`).

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    "I":   (x123, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x123, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x123, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}

rows = []
for name, (xs, ys) in anscombe.items():
    xs, ys = np.array(xs, float), np.array(ys, float)
    # TODO: mean x, var x, mean y, var y, Pearson, slope, Spearman
    rows.append((name, ...))
# pd.DataFrame(rows, columns=[...]).round(3)

### ✏️ Задание 6.3. Корреляции в реальных данных

Датасет California Housing из `sample_data` в Colab (как на семинаре 3).

1. Постройте матрицы корреляций Пирсона и Спирмена для колонок `median_income`, `median_house_value`, `housing_median_age`, `total_rooms`, `population`, `households`. Нарисуйте их через `plt.imshow` или выведите таблицей.
2. Какие пары сильнее всего коррелированы? Объясните, почему `total_rooms`, `population` и `households` так сильно связаны (подсказка: размер района — общая причина).
3. Создайте признак `rooms_per_household = total_rooms / households`. Как меняется его корреляция с `median_house_value` по сравнению с `total_rooms`?
4. Найдите наклон линейной регрессии `median_house_value` на `median_income` по формуле $b=\rho\,s_y/s_x$ и сравните с `np.polyfit(x, y, 1)`. Какую долю дисперсии стоимости объясняет доход?

In [ ]:
import os

PATHS = ["/content/sample_data/california_housing_train.csv",
         "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"]
df = None
for path in PATHS:
    try:
        if path.startswith("http") or os.path.exists(path):
            df = pd.read_csv(path)
            print("загружено:", path)
            break
    except Exception as exc:
        print("не получилось:", path, exc)
if df is None:   # запасной вариант без интернета: синтетические данные похожей структуры
    print("Датасет недоступен, используем синтетические данные")
    n = 17_000
    size = rng.lognormal(np.log(1100), 0.6, n)                      # «размер района»
    households = np.round(size * rng.lognormal(0, 0.15, n))
    income = rng.lognormal(np.log(3.5), 0.45, n)
    df = pd.DataFrame({
        "median_income": income,
        "housing_median_age": rng.integers(1, 53, n),
        "households": households,
        "population": np.round(households * rng.lognormal(np.log(2.9), 0.25, n)),
        "total_rooms": np.round(households * (3 + 0.6 * income) * rng.lognormal(0, 0.2, n)),
    })
    df["median_house_value"] = np.clip(40_000 * income + rng.normal(0, 70_000, n) + 30_000, 15_000, 500_001)

cols = ["median_income", "median_house_value", "housing_median_age", "total_rooms", "population", "households"]
df[cols].describe().round(2)

In [ ]:
# TODO: 1–4
corr_pearson = ...
corr_spearman = ...

*Вопрос.* `median_house_value` обрезана сверху на 500 001 (семинар 3). Как обрезка влияет на корреляцию с доходом и на наклон регрессии — завышает или занижает их?

---
## Итоги

- $EX$ — центр масс; среднее выборки сходится к нему (ЗБЧ), но только если $EX$ существует (Коши — контрпример).
- Линейность ожидания работает для зависимых величин; $E(XY)=EX\,EY$ и $D(X+Y)=DX+DY$ — только при независимости (некоррелированности).
- Метод индикаторов превращает трудные ожидания в сумму вероятностей.
- $EX^2-(EX)^2$ численно неустойчива; `np.var` делит на $n$, `pandas` — на $n-1$.
- Чебышёв даёт гарантию для любого распределения, но правило $|z|>3$ работает только для почти нормальных данных.
- Погрешность среднего падает как $\sigma/\sqrt n$.
- Ковариация управляет риском суммы; корреляция ловит только линейную связь — всегда смотрите на графики.

**Домашнее задание** — отдельный ноутбук `hw04-expectation-variance-itmo.ipynb` (условия и порядок сдачи — внутри).